In [1]:
import torch
from torch import nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from mlpcontrollermodel import CarDataset, load_data_from_csvs, MLPControllerModel

In [2]:
data = load_data_from_csvs('./data')
# Split the data into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [3]:
# Create PyTorch datasets and loaders
train_dataset = CarDataset(train_data)
test_dataset = CarDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=15000, shuffle=True, prefetch_factor=8, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=15000, shuffle=False)

In [4]:
print(len(train_loader))

107


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MLPControllerModel().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.2)

In [6]:
# Training loop
num_epochs = 30

# Interval of steps to print status
print_every = 10

model.train()
for epoch in range(num_epochs):
    step_loss = 0.0  # Accumulate loss for printing every n steps
    epoch_loss = 0.0  # Accumulate loss for the entire epoch

    for i, (features, labels) in enumerate(train_loader):
        # Move data to the same device as the model
        features, steerCommand = features.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(features)
        loss = criterion(outputs, steerCommand.view(-1, 1))
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate running loss
        epoch_loss += loss.item()
        step_loss += loss.item()

        # Print loss every n steps
        if (i + 1) % print_every == 0:
            avg_loss_step = step_loss / print_every
            print(f'Epoch {epoch+1}/{num_epochs}, Step {i+1}/{len(train_loader)}, Loss: {avg_loss_step:.4f}')
            step_loss = 0.0  # Reset running loss after printing


    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}')

Epoch 1/30, Step 10/107, Loss: 0.9818
Epoch 1/30, Step 20/107, Loss: 0.2066
Epoch 1/30, Step 30/107, Loss: 0.1075
Epoch 1/30, Step 40/107, Loss: 0.0706
Epoch 1/30, Step 50/107, Loss: 0.0550
Epoch 1/30, Step 60/107, Loss: 0.0462
Epoch 1/30, Step 70/107, Loss: 0.0430
Epoch 1/30, Step 80/107, Loss: 0.0397
Epoch 1/30, Step 90/107, Loss: 0.0338
Epoch 1/30, Step 100/107, Loss: 0.0297
Epoch 1/30, Loss: 0.1527
Epoch 2/30, Step 10/107, Loss: 0.0303
Epoch 2/30, Step 20/107, Loss: 0.0287
Epoch 2/30, Step 30/107, Loss: 0.0277
Epoch 2/30, Step 40/107, Loss: 0.0269
Epoch 2/30, Step 50/107, Loss: 0.0263
Epoch 2/30, Step 60/107, Loss: 0.0258
Epoch 2/30, Step 70/107, Loss: 0.0258
Epoch 2/30, Step 80/107, Loss: 0.0256
Epoch 2/30, Step 90/107, Loss: 0.0252
Epoch 2/30, Step 100/107, Loss: 0.0253
Epoch 2/30, Loss: 0.0267
Epoch 3/30, Step 10/107, Loss: 0.0248
Epoch 3/30, Step 20/107, Loss: 0.0244
Epoch 3/30, Step 30/107, Loss: 0.0243
Epoch 3/30, Step 40/107, Loss: 0.0242
Epoch 3/30, Step 50/107, Loss: 0.024

In [7]:
print(features)

tensor([[ 2.3576e+01, -1.1919e-01, -4.3215e-02, -3.0147e-01],
        [ 2.1871e+01,  9.2118e-01,  3.1078e-02, -4.7164e-02],
        [ 2.7896e+01, -5.1033e-02,  3.8516e-02, -1.0825e-01],
        ...,
        [ 1.8308e+01, -4.4447e-02, -1.4284e-02,  1.6674e+00],
        [ 1.7371e+01,  1.7143e-01,  1.3524e-02,  9.5469e-04],
        [ 3.0445e+01, -4.8502e-02,  4.8501e-03,  4.7015e-02]], device='cuda:0')


In [8]:
torch.save(model, './models/mlp_controller_model.pth')